# Probability Mathematics and Information Theory

Notebook for **Chapter 15** of the *math* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/math/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 Bridge Recap and Expectation Algebra

In [ ]:
import numpy as np
x = np.arange(1, 7)
p = np.full(6, 1/6)
mean = np.sum(x * p)
ex2 = np.sum(x**2 * p)          # LOTUS: weight g(x) = x^2 by p(x)
print(mean, round(ex2, 4), round(ex2 - mean**2, 4))
rng = np.random.default_rng(0)
rolls = rng.integers(1, 7, 200000)
print(round(rolls.mean(), 3), round(rolls.var(), 3))

3.5 15.1667 2.9167
3.5 2.918


In [ ]:
import numpy as np
rng = np.random.default_rng(1)
n = 400000
express = rng.random(n) < 0.4
t = np.where(express, rng.normal(20, 5, n), rng.normal(50, 10, n))
print(round(t.mean(), 2), round(t.var(), 1), round(t.std(), 2))

37.97 286.0 16.91


## 15.2 Covariance, Correlation, MGFs and Transformations

In [ ]:
import numpy as np
x = np.array([1, 2, 3, 4.0]); y = np.array([2, 3, 5, 6.0])
print(round(np.mean((x - x.mean()) * (y - y.mean())), 4))   # divide by n
print(round(np.cov(x, y)[0, 1], 4))                        # divide by n - 1
print(np.corrcoef(x, y)[0, 1].round(4))

1.75
2.3333
0.9899


In [ ]:
import numpy as np
S = np.array([[4, 2.4], [2.4, 2.0]])
a = np.array([1, -1.0])
print(round(a @ S @ a, 4))
print(np.linalg.eigvalsh(S).round(4))
rng = np.random.default_rng(2)
X = rng.multivariate_normal([0, 0], S, 200000)
print(np.round(np.cov(X.T), 2))

1.2
[0.4 5.6]
[[4.  2.4]
 [2.4 2. ]]


In [ ]:
import numpy as np
from scipy.integrate import quad
lam, t = 2.0, 0.5
val, err = quad(lambda x: np.exp(t * x) * lam * np.exp(-lam * x), 0, 60)
print(round(val, 4), round(lam / (lam - t), 4))
rng = np.random.default_rng(3)
total = rng.exponential(1 / lam, (300000, 3)).sum(axis=1)
print(round(total.mean(), 3), round(total.var(), 3))

1.3333 1.3333
1.502 0.753


In [ ]:
import numpy as np
rng = np.random.default_rng(4)
z = rng.standard_normal(500000)
y = np.exp(z)                       # lognormal with mu=0, sigma=1
print(round(y.mean(), 3), round(np.median(y), 3), round(np.exp(0.5), 4))
k = 5
chi = (rng.standard_normal((500000, k)) ** 2).sum(axis=1)
print(round(chi.mean(), 3), round(chi.var(), 3))

1.646 1.0 1.6487
4.992 9.971


## 15.3 The Multivariate Normal and the Gaussian Integral

In [ ]:
import numpy as np
mu = np.array([2.0, 1.0]); S = np.array([[4, 2.4], [2.4, 2.0]])
Sinv = np.linalg.inv(S)
for x in ([5, 3], [0, 4]):
    d = np.array(x) - mu
    m2 = d @ Sinv @ d
    print(x, round(m2, 4), round(np.sqrt(m2), 4), round(np.linalg.norm(d), 4))
d = np.array([3.0, 2.0])
dens = np.exp(-0.5 * d @ Sinv @ d) / (2 * np.pi * np.sqrt(np.linalg.det(S)))
print(round(dens, 5))
vals, vecs = np.linalg.eigh(S)
print(vals.round(3), np.abs(vecs).round(4).tolist())

[5, 3] 2.3214 1.5236 3.6056
[0, 4] 32.5 5.7009 3.6056
0.03331
[0.4 5.6] [[0.5547, 0.8321], [0.8321, 0.5547]]


In [ ]:
import numpy as np
rng = np.random.default_rng(5)
mu = np.array([2.0, 1.0]); S = np.array([[4, 2.4], [2.4, 2.0]])
X = rng.multivariate_normal(mu, S, 1000000)
near = X[np.abs(X[:, 1] - 3) < 0.02, 0]       # samples with X2 close to 3
print(round(near.mean(), 2), round(near.var(), 2))
print(round((X[:, 0] + X[:, 1]).mean(), 2), round((X[:, 0] + X[:, 1]).var(), 2))

4.41 1.14
3.0 10.79


In [ ]:
import numpy as np
S = np.array([[4, 2.4], [2.4, 2.0]])
L = np.linalg.cholesky(S)
print(L.round(4))
print((L @ L.T).round(4).tolist())
rng = np.random.default_rng(6)
Z = rng.standard_normal((200000, 2))
X = np.array([2, 1.0]) + Z @ L.T          # rows are samples
print(X.mean(axis=0).round(2), np.cov(X.T).round(2).tolist())
# whitening: undo the transform, M^2 = squared length of z
W = np.linalg.solve(L, (X - [2, 1.0]).T).T
m2 = (W**2).sum(axis=1)
print(round(np.mean(m2 <= 5.9915), 3))   # chi-square(2) 95 percent point

[[2.     0.    ]
 [1.2    0.7483]]
[[4.0, 2.4], [2.4, 2.0]]
[2. 1.] [[4.0, 2.4], [2.4, 2.0]]
0.95


In [ ]:
import numpy as np
from scipy.integrate import quad
val, _ = quad(lambda x: np.exp(-x**2), -np.inf, np.inf)
print(round(val**2, 6), round(np.pi, 6))
val2, _ = quad(lambda x: np.exp(-x**2 / 2), -np.inf, np.inf)
print(round(val2, 6), round(np.sqrt(2 * np.pi), 6))

3.141593 3.141593
2.506628 2.506628


## 15.4 Inequalities, the Law of Large Numbers and the CLT

In [ ]:
import numpy as np
from scipy.stats import binom
n, t = 1000, 0.05
print(round(0.25 / (n * t**2), 4))
print(round(2 * np.exp(-2 * n * t**2), 5))
print(round(2 * binom.sf(549, n, 0.5), 5))     # P(S >= 550) twice

0.1
0.01348
0.00173


In [ ]:
import numpy as np
from scipy.stats import norm, gamma
n = 30
sd = 1 / np.sqrt(n)
print(round(sd, 4), round(norm.cdf((1.3 - 1) / sd), 4), round(gamma.cdf(1.3 * n, n), 4))
rng = np.random.default_rng(7)
means = rng.exponential(1.0, (100000, n)).mean(axis=1)
print(round(means.mean(), 3), round(means.std(), 3), round(np.mean(means <= 1.3), 4))

0.1826 0.9498 0.9409
1.0 0.182 0.9412


In [ ]:
import numpy as np
rng = np.random.default_rng(8)
for d in (2, 100, 1000):
    norms = np.linalg.norm(rng.standard_normal((20000, d)), axis=1)
    print(d, round(norms.mean(), 2), round(norms.std(), 3), round(np.sqrt(d), 2))
a = rng.standard_normal((2000, 1000)); a /= np.linalg.norm(a, axis=1, keepdims=True)
cos = a[:1000] @ a[1000:].T
print(round(np.abs(cos).mean(), 4))

2 1.25 0.65 1.41
100 9.97 0.708 10.0
1000 31.62 0.713 31.62
0.0252


## 15.5 Entropy, Cross-Entropy, KL Divergence and Mutual Information

In [ ]:
import numpy as np
p = np.array([0.5, 0.25, 0.25]); q = np.array([0.4, 0.4, 0.2])
H = -np.sum(p * np.log2(p))
CE = -np.sum(p * np.log2(q))
KL = np.sum(p * np.log2(p / q))
print(round(H, 4), round(CE, 4), round(KL, 4), round(H + KL, 4))

1.5 1.5719 0.0719 1.5719


In [ ]:
import numpy as np
y = np.array([1, 0, 1]); q = np.array([0.8, 0.3, 0.6])
loss = -(y * np.log(q) + (1 - y) * np.log(1 - q))
print(loss.round(4), round(loss.mean(), 4))
# confident and wrong
print(round(-np.log(0.001), 3), round(-np.log(0.5), 3))
# softmax gradient check by finite differences
z = np.array([2.0, 1.0, 0.0]); p = np.array([0.0, 1.0, 0.0])
def L(z):
    q = np.exp(z) / np.exp(z).sum(); return -np.sum(p * np.log(q))
q = np.exp(z) / np.exp(z).sum()
num = np.array([(L(z + 1e-6 * np.eye(3)[k]) - L(z - 1e-6 * np.eye(3)[k])) / 2e-6 for k in range(3)])
print((q - p).round(4), num.round(4))

[0.2231 0.3567 0.5108] 0.3635
6.908 0.693
[ 0.6652 -0.7553  0.09  ] [ 0.6652 -0.7553  0.09  ]


In [ ]:
import numpy as np
J = np.array([[0.20, 0.20], [0.06, 0.54]])    # rows: spam, ham; columns: free, no free
px = J.sum(axis=1); py = J.sum(axis=0)
mi = np.sum(J * np.log2(J / np.outer(px, py)))
print(round(mi, 4))
# if the two were independent, the joint would be the product of marginals: MI is zero
Jind = np.outer(px, py)
print(round(np.sum(Jind * np.log2(Jind / np.outer(px, py))), 4))

0.1453
0.0


In [ ]:
import numpy as np
rng = np.random.default_rng(9)
n = 20000
y = rng.integers(0, 2, n)
f_signal = (rng.random(n) < np.where(y == 1, 0.8, 0.3)).astype(int)   # depends on y
f_noise = rng.integers(0, 2, n)                                           # independent of y
def mi(a, b):
    J = np.zeros((2, 2))
    for i in (0, 1):
        for j in (0, 1): J[i, j] = np.mean((a == i) & (b == j))
    P = np.outer(J.sum(1), J.sum(0))
    return np.sum(J * np.log2(J / P))
print(round(mi(f_signal, y), 4), round(mi(f_noise, y), 5))

0.1766 0.0


In [ ]:
import numpy as np
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance
p = np.array([1.0, 0.0, 0.0, 0.0]); q1 = np.array([0.0, 1.0, 0.0, 0.0]); q3 = np.array([0.0, 0.0, 0.0, 1.0])
print(round(jensenshannon(p, q1, base=2)**2, 4), round(jensenshannon(p, q3, base=2)**2, 4))
pos = np.arange(4)
print(wasserstein_distance(pos, pos, p, q1), wasserstein_distance(pos, pos, p, q3))

1.0 1.0
1.0 3.0


In [ ]:
import numpy as np
def H(counts):
    p = np.array(counts) / sum(counts); p = p[p > 0]
    return -np.sum(p * np.log2(p))
parent = H([6, 4]); a = H([4, 1]); b = H([2, 3])
gain = parent - (5/10 * a + 5/10 * b)
print(round(parent, 4), round(a, 4), round(b, 4), round(gain, 4))

0.971 0.7219 0.971 0.1245


In [ ]:
import numpy as np
probs = np.array([0.5, 0.25, 0.1, 0.4])
h_bits = -np.mean(np.log2(probs))
print(round(h_bits, 4), round(2**h_bits, 3), round(np.exp(-np.mean(np.log(probs))), 3))

1.911 3.761 3.761


## 15.6 Markov Chains

In [ ]:
import numpy as np
P = np.array([[0.9, 0.1], [0.5, 0.5]])
print(np.linalg.matrix_power(P, 2).round(4).tolist())
vals, vecs = np.linalg.eig(P.T)          # left eigenvectors of P
k = np.argmax(vals.real)
pi = vecs[:, k].real; pi = pi / pi.sum()
print(pi.round(4), np.sort(vals.real).round(2))
start = np.array([0.0, 1.0])             # start Dormant
for n in (1, 3, 10):
    print(n, (start @ np.linalg.matrix_power(P, n)).round(4))

[[0.86, 0.14], [0.7, 0.3]]
[0.8333 0.1667] [0.4 1. ]
1 [0.5 0.5]
3 [0.78 0.22]
10 [0.8332 0.1668]


In [ ]:
import numpy as np
Q = np.array([[0.5, 0.3], [0.0, 0.8]]); R = np.array([[0.2, 0.0], [0.15, 0.05]])
N = np.linalg.inv(np.eye(2) - Q)
print(N.round(3).tolist(), N.sum(axis=1).round(3).tolist())
print((N @ R).round(3).tolist())

[[2.0, 3.0], [0.0, 5.0]] [5.0, 5.0]
[[0.85, 0.15], [0.75, 0.25]]


## 15.7 Monte Carlo, Importance Sampling and MCMC

In [ ]:
import numpy as np
rng = np.random.default_rng(10)
N = 10000
g = np.exp(-rng.random(N)**2)
est = g.mean(); se = g.std(ddof=1) / np.sqrt(N)
print(round(est, 4), round(se, 4), round(g.std(), 3))
print(abs(est - 0.746824) < 3 * se)

0.7455 0.002 0.201
True


In [ ]:
import numpy as np
from scipy.stats import norm
rng = np.random.default_rng(11)
N = 10000
plain = np.mean(rng.standard_normal(N) > 4)
y = rng.normal(4, 1, N)
w = np.exp(-4 * y + 8)
imp = np.mean((y > 4) * w)
print(plain, f"{imp:.3e}", f"{norm.sf(4):.3e}")

0.0 3.270e-05 3.167e-05


In [ ]:
import numpy as np

def target(x):
    # unnormalised is fine; this one happens to be normalised
    a = 0.3 * np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)
    b = 0.7 * np.exp(-(x - 4)**2 / (2 * 0.8**2)) / (0.8 * np.sqrt(2 * np.pi))
    return a + b

def metropolis(target, x0, n_steps, step, rng):
    x = x0
    chain = np.empty(n_steps)
    accepted = 0
    for i in range(n_steps):
        prop = x + rng.normal(0, step)          # symmetric random-walk proposal
        ratio = target(prop) / target(x)        # constants cancel here
        if rng.random() < ratio:                # accept with prob min(1, ratio)
            x = prop
            accepted += 1
        chain[i] = x                            # record (repeat x on rejection)
    return chain, accepted / n_steps

rng = np.random.default_rng(12)
print(round(target(3.0), 4), round(target(4.0), 4), round(target(1.5), 4))
chain, rate = metropolis(target, 0.0, 20000, 1.5, rng)
kept = chain[1000:]                             # drop burn-in
print("acceptance", round(rate, 2))
print("mean", round(kept.mean(), 2), "target mean 2.8")
print("share above 2", round(np.mean(kept > 2), 2))

0.1611 0.3491 0.0415
acceptance 0.61
mean 2.77 target mean 2.8
share above 2 0.69


## 15.8 The Expert Layer: Exponential Families, Fisher Information, ELBO and Reparameterization

In [ ]:
import numpy as np
rng = np.random.default_rng(13)
p, n = 0.3, 100
phat = rng.binomial(n, p, 200000) / n
print(round(phat.var(), 5), round(p * (1 - p) / n, 5), round(1 / (p * (1 - p)), 3))

0.0021 0.0021 4.762


In [ ]:
import numpy as np
def softmax(z, T=1.0):
    e = np.exp((z - z.max()) / T)             # subtract max for stability
    return e / e.sum()
z = np.array([2.0, 1.0, 0.0])
for T in (1.0, 0.5, 2.0, 10.0):
    q = softmax(z, T)
    print(T, q.round(4), round(-np.sum(q * np.log(q)), 4))
print(round(np.log(3), 4))

1.0 [0.6652 0.2447 0.09  ] 0.8324
0.5 [0.8668 0.1173 0.0159] 0.4411
2.0 [0.5065 0.3072 0.1863] 1.0202
10.0 [0.3672 0.3322 0.3006] 1.0953
1.0986


In [ ]:
import numpy as np
rng = np.random.default_rng(14)
mu, sigma, N = 1.5, 0.8, 100000
eps = rng.standard_normal(N)
z = mu + sigma * eps
g_rep = 2 * z                              # derivative of z^2 pushed through z = mu + sigma*eps
g_score = z**2 * (z - mu) / sigma**2       # score-function estimator
print(round(g_rep.mean(), 3), round(g_rep.std(), 3))
print(round(g_score.mean(), 3), round(g_score.std(), 3))

3.0 1.6
2.992 6.947


## Chapter recap

## Practice set

Problem 15.10.1 A game pays 10 with probability 0.2, 0 with probability 0.5 and −4 with probability 0.3. Find the expected payout and its variance.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/math/ch15.html